# Notebook 4 — Entrenamiento, Predicción y Simulación

**Objetivo**: entrenar el modelo de goles, elegir la familia ganadora según la evidencia del
Notebook 3 (no según un solo torneo), y simular el propio Mundial de dos formas complementarias:

- **4.1 a 4.12**: el pipeline de predicción "en vivo" -- reentrenamiento walk-forward, cada
  ronda eliminatoria evaluada con el modelo tal como estaba ANTES de conocer su resultado, y
  solo después incorporando ese resultado real para la ronda siguiente.
- **4.13 en adelante**: simulación Montecarlo del cuadro completo, miles de veces, para obtener
  la probabilidad de cada selección de llegar a cada ronda -- parametrizable para verla "como se
  veía el día antes del Mundial" (`pre_mundial`) o "con todo lo jugado hasta hoy" (`actual`).

Entrada: `data/processed/partidos_features.csv` (Notebook 2), conclusiones del Notebook 3
(familia ganadora) y `data/raw/wc2026_calendario.json` (estructura del cuadro).

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import json
import lightgbm as lgb
import networkx as nx
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)  # el notebook ya imprime su propio resumen

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_MODELOS = DIR_RAIZ / "models"
# Artefactos finales para el portfolio (predicciones, comparaciones) van aquí,
# no en data/processed/ -- ese está en .gitignore porque se regenera con solo
# re-ejecutar el pipeline, pero results/ SÍ se versiona en git.
DIR_RESULTS = DIR_RAIZ / "results"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

SEMILLA = 42
# Rejilla 0..8: ahora la matriz decide SIEMPRE quién avanza (4.6), incluidos
# partidos con ventaja grande (lambda ~4), no solo cruces parejos -- con 0..5
# se perdía masa de probabilidad real en esos casos.
MAX_GOLES_MATRIZ = 8

# Conclusiones del Notebook 3 (obtenidas allí con evidencia; aquí se consumen
# como constantes documentadas, no se repite el análisis).
FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
FEATURES_MODELO = [
    "elo_diff", "tendencia_elo_local", "tendencia_elo_visitante",
    "dif_forma_gf_5", "dif_forma_gf_10", "dif_racha_5", "dif_racha_10",
    "dias_descanso_local", "dias_descanso_visitante",
    "es_neutral", "h2h_puntos_prom", "h2h_dif_goles_prom",
]

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
print(f"Partidos desde {FECHA_CORTE_HISTORICO.date()}: {len(df):,}")

Partidos desde 1990-01-01: 32,382


## 4.1 Detectar fase de grupos vs. eliminatorias del Mundial 2026

`results.csv` no trae una columna de "ronda" explícita para el Mundial en curso. La
fase de grupos SÍ se puede reconstruir sin ambigüedad: dentro de las selecciones que
han jugado el Mundial 2026, un grupo real es un conjunto de 4 equipos que se han
enfrentado entre sí TODOS contra TODOS (round-robin completo = camarilla de 4 en el
grafo de "quién jugó contra quién"). Un cruce de eliminatoria aislado entre dos
selecciones de grupos distintos no basta para simular una camarilla falsa, así que el
método es robusto aunque ya se hayan jugado partidos de eliminatoria — no hace falta
ninguna fuente externa ni fecha de corte arbitraria para separar ambas fases.

In [2]:
def detectar_fase_grupos(df_mundial: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Separa los partidos del Mundial 2026 en (fase_de_grupos, eliminatorias).

    Construye el grafo de enfrentamientos y busca camarillas de 4 (grupos
    reales); cualquier partido que no sea interno a una de esas camarillas
    es, por descarte, de eliminatoria directa.
    '''
    grafo = nx.Graph()
    for _, fila in df_mundial.iterrows():
        grafo.add_edge(fila["equipo_local"], fila["equipo_visitante"])

    camarillas_de_4 = [c for c in nx.enumerate_all_cliques(grafo) if len(c) == 4]
    usados: set[str] = set()
    grupos: list[frozenset[str]] = []
    for c in camarillas_de_4:
        if not (set(c) & usados):
            grupos.append(frozenset(c))
            usados.update(c)

    def _mismo_grupo(fila) -> bool:
        par = {fila["equipo_local"], fila["equipo_visitante"]}
        return any(par <= g for g in grupos)

    es_fase_grupos = df_mundial.apply(_mismo_grupo, axis=1)
    print(f"Grupos detectados: {len(grupos)} ({sum(len(g) for g in grupos)} selecciones)")
    print(f"Partidos de fase de grupos: {es_fase_grupos.sum()} | de eliminatoria: {(~es_fase_grupos).sum()}")
    return df_mundial[es_fase_grupos].copy(), df_mundial[~es_fase_grupos].copy()


df_mundial_2026 = df[(df["torneo"] == "FIFA World Cup") & (df["fecha"].dt.year == 2026)]
df_grupos, df_eliminatoria = detectar_fase_grupos(df_mundial_2026)

Grupos detectados: 12 (48 selecciones)
Partidos de fase de grupos: 72 | de eliminatoria: 24


## 4.2 Split temporal

- **Entrenamiento**: todo el histórico (desde 1990) hasta el **10 de junio de 2026**,
  el día antes de que arrancara el Mundial — el modelo no ve ni un solo partido del
  torneo antes de predecirlo.
- **Test inicial**: los partidos de **fase de grupos ya jugados**.

In [3]:
FECHA_INICIO_MUNDIAL = pd.Timestamp("2026-06-11")

df_train_inicial = df[(df["fecha"] < FECHA_INICIO_MUNDIAL) & df["jugado"]].copy()
df_test_grupos = df_grupos[df_grupos["jugado"]].copy()

print(f"Entrenamiento inicial: {len(df_train_inicial):,} partidos (hasta {FECHA_INICIO_MUNDIAL.date()})")
print(f"Test (fase de grupos ya jugada): {len(df_test_grupos):,} partidos")

assert df_train_inicial["fecha"].max() < FECHA_INICIO_MUNDIAL, "Fuga temporal: hay partidos del Mundial en el train" 

Entrenamiento inicial: 32,286 partidos (hasta 2026-06-11)
Test (fase de grupos ya jugada): 72 partidos


## 4.3 Modelos base de goles: Poisson GLM vs. Gradient Boosting

Se entrena un modelo INDEPENDIENTE para los goles del local y otro para los del
visitante — mezclar ambos en un único target no tiene sentido (son dos procesos con
features "local"/"visitante" invertidas). Dos familias candidatas:

- **Modelo A — Regresión de Poisson (GLM)**: el fútbol es el caso de libro de un
  proceso de conteo (goles en un tiempo fijo, sucesos raros e independientes entre
  sí); un GLM de Poisson lo modela de forma nativa y sus predicciones son siempre
  tasas positivas, listas para la matriz de probabilidad conjunta de más abajo.
- **Modelo B — LightGBM con objetivo Poisson**: captura relaciones no lineales e
  interacciones entre features que un GLM lineal no puede, al coste de ser una caja
  más negra y de necesitar más datos para no sobreajustar.

In [4]:
def entrenar_poisson_glm(X: pd.DataFrame, y: pd.Series, alpha: float = 0.5):
    '''Regresión de Poisson (GLM) — alpha es la regularización L2; un valor
    demasiado alto aplana las predicciones hacia la media global (síntoma
    clásico: el modelo predice casi el mismo marcador para cualquier partido).

    Escalado obligatorio: `elo_diff` se mueve en el orden de los cientos
    mientras que `dif_racha_5` lo hace en unidades — sin normalizar, el
    solver del GLM queda mal condicionado (produce advertencias de overflow
    en el gradiente) y los coeficientes dejan de ser comparables entre sí.
    '''
    return make_pipeline(StandardScaler(), PoissonRegressor(alpha=alpha, max_iter=1000)).fit(X, y)


def entrenar_lightgbm_poisson(X: pd.DataFrame, y: pd.Series, params: dict | None = None) -> lgb.LGBMRegressor:
    '''LightGBM con objetivo Poisson (tasas de gol, nunca negativas) en vez de
    error cuadrático — con error cuadrático la varianza real del fútbol (mayor
    cuantos más goles se esperan) queda mal representada.
    '''
    params_finales = {"objective": "poisson", "random_state": SEMILLA, "verbosity": -1, "n_estimators": 300}
    params_finales.update(params or {})
    return lgb.LGBMRegressor(**params_finales).fit(X, y)


def entrenar_xgboost_poisson(X: pd.DataFrame, y: pd.Series, params: dict | None = None) -> xgb.XGBRegressor:
    '''XGBoost con el mismo objetivo Poisson que LightGBM -- añadido tras el
    backtesting del Notebook 3: contra 2010/2014/2018/2022, XGBoost gana en
    RMSE y LogLoss en 3 de 4 Mundiales (GLM solo ganaba en 2026, el único
    torneo que este notebook había mirado hasta ahora). Ver
    `results/backtesting_mundiales.csv`.
    '''
    params_finales = {"objective": "count:poisson", "random_state": SEMILLA, "verbosity": 0, "n_estimators": 300}
    params_finales.update(params or {})
    return xgb.XGBRegressor(**params_finales).fit(X, y)


X_train_inicial = df_train_inicial[FEATURES_MODELO]
X_test_grupos = df_test_grupos[FEATURES_MODELO]

modelos_glm = {
    "local": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_local"]),
    "visitante": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_visitante"]),
}
modelos_lgbm_base = {
    "local": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_local"]),
    "visitante": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_visitante"]),
}
modelos_xgb_base = {
    "local": entrenar_xgboost_poisson(X_train_inicial, df_train_inicial["goles_local"]),
    "visitante": entrenar_xgboost_poisson(X_train_inicial, df_train_inicial["goles_visitante"]),
}
print("Modelos base entrenados (Poisson GLM, LightGBM y XGBoost sin ajustar).")

Modelos base entrenados (Poisson GLM, LightGBM y XGBoost sin ajustar).


## 4.4 Optimización de hiperparámetros con Optuna (LightGBM)

In [5]:
def objetivo_optuna(trial: optuna.Trial, X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> float:
    '''Función objetivo: RMSE medio en validación cruzada TEMPORAL (nunca un
    K-Fold barajado — en series temporales eso entrenaría con 2025 para
    predecir 2010, inflando la métrica de forma escandalosa).
    '''
    params = {
        "objective": "poisson",
        "random_state": SEMILLA,
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    cv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for idx_train, idx_val in cv.split(X):
        modelo = lgb.LGBMRegressor(**params).fit(X.iloc[idx_train], y.iloc[idx_train])
        pred = np.clip(modelo.predict(X.iloc[idx_val]), 0, None)
        rmses.append(np.sqrt(mean_squared_error(y.iloc[idx_val], pred)))
    return float(np.mean(rmses))


def optimizar_lightgbm(X: pd.DataFrame, y: pd.Series, n_trials: int = 40, nombre: str = "") -> optuna.Study:
    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
    estudio.optimize(lambda trial: objetivo_optuna(trial, X, y), n_trials=n_trials, show_progress_bar=False)
    print(f"[{nombre}] Mejor RMSE (CV temporal): {estudio.best_value:.4f}")
    print(f"[{nombre}] Mejores hiperparámetros: {estudio.best_params}")
    return estudio


estudio_local = optimizar_lightgbm(X_train_inicial, df_train_inicial["goles_local"], nombre="goles_local")
estudio_visitante = optimizar_lightgbm(X_train_inicial, df_train_inicial["goles_visitante"], nombre="goles_visitante")

modelos_lgbm_optuna = {
    "local": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_local"], estudio_local.best_params),
    "visitante": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_visitante"], estudio_visitante.best_params),
}


def objetivo_optuna_xgb(trial: optuna.Trial, X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> float:
    """Mismo espacio de busqueda que el Notebook 3 para XGBoost, con CV
    temporal -- nunca K-Fold barajado, mismo motivo que `objetivo_optuna`."""
    params = {
        "objective": "count:poisson",
        "random_state": SEMILLA,
        "verbosity": 0,
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    cv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for idx_train, idx_val in cv.split(X):
        modelo = xgb.XGBRegressor(**params).fit(X.iloc[idx_train], y.iloc[idx_train])
        pred = np.clip(modelo.predict(X.iloc[idx_val]), 0, None)
        rmses.append(np.sqrt(mean_squared_error(y.iloc[idx_val], pred)))
    return float(np.mean(rmses))


def optimizar_xgboost(X: pd.DataFrame, y: pd.Series, n_trials: int = 40, nombre: str = "") -> optuna.Study:
    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
    estudio.optimize(lambda trial: objetivo_optuna_xgb(trial, X, y), n_trials=n_trials, show_progress_bar=False)
    print(f"[{nombre}] Mejor RMSE (CV temporal): {estudio.best_value:.4f}")
    print(f"[{nombre}] Mejores hiperparametros: {estudio.best_params}")
    return estudio


estudio_local_xgb = optimizar_xgboost(X_train_inicial, df_train_inicial["goles_local"], nombre="goles_local (xgb)")
estudio_visitante_xgb = optimizar_xgboost(X_train_inicial, df_train_inicial["goles_visitante"], nombre="goles_visitante (xgb)")

modelos_xgb_optuna = {
    "local": entrenar_xgboost_poisson(X_train_inicial, df_train_inicial["goles_local"], estudio_local_xgb.best_params),
    "visitante": entrenar_xgboost_poisson(X_train_inicial, df_train_inicial["goles_visitante"], estudio_visitante_xgb.best_params),
}

[goles_local] Mejor RMSE (CV temporal): 1.5327
[goles_local] Mejores hiperparámetros: {'n_estimators': 343, 'num_leaves': 23, 'learning_rate': 0.02218680407887412, 'min_child_samples': 61, 'reg_lambda': 0.00567989618228988, 'subsample': 0.9259308962023542, 'colsample_bytree': 0.8536051970294887}


[goles_visitante] Mejor RMSE (CV temporal): 1.2229
[goles_visitante] Mejores hiperparámetros: {'n_estimators': 546, 'num_leaves': 51, 'learning_rate': 0.010055066590996429, 'min_child_samples': 75, 'reg_lambda': 2.027854767488019, 'subsample': 0.6225953269813417, 'colsample_bytree': 0.6427551274106433}


[goles_local (xgb)] Mejor RMSE (CV temporal): 1.5315
[goles_local (xgb)] Mejores hiperparametros: {'n_estimators': 253, 'max_depth': 5, 'learning_rate': 0.03414844546169164, 'min_child_weight': 7, 'reg_lambda': 0.1300440138220517, 'subsample': 0.7357010062288922, 'colsample_bytree': 0.7603864950608998}


[goles_visitante (xgb)] Mejor RMSE (CV temporal): 1.2243
[goles_visitante (xgb)] Mejores hiperparametros: {'n_estimators': 279, 'max_depth': 3, 'learning_rate': 0.04418272459545804, 'min_child_weight': 2, 'reg_lambda': 0.0032952954237292216, 'subsample': 0.9373775148089885, 'colsample_bytree': 0.6537135219185505}


## 4.5 Evaluación en la fase de grupos: GLM vs. LightGBM afinado

Dos métricas, dos preguntas distintas: **RMSE** sobre los goles exactos, y **LogLoss**
sobre el resultado 1X2 derivado de la rejilla conjunta de Poisson (más abajo) — un
modelo puede acertar el marcador y aun así dar probabilidades mal calibradas, o
viceversa.

In [6]:
def log_verosimilitud_dixon_coles(rho: float, lam: np.ndarray, mu: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    '''Log-verosimilitud de los marcadores observados (x, y) bajo la corrección
    de Dixon-Coles, con las lambdas (lam, mu) ya estimadas por el GLM -- se usa
    solo para calibrar rho por rejilla, con las tasas de gol ya fijas (el
    procedimiento en dos pasos del propio paper de 1997).'''
    tau = np.ones_like(lam)
    es_0_0 = (x == 0) & (y == 0)
    es_0_1 = (x == 0) & (y == 1)
    es_1_0 = (x == 1) & (y == 0)
    es_1_1 = (x == 1) & (y == 1)
    tau[es_0_0] = 1 - lam[es_0_0] * mu[es_0_0] * rho
    tau[es_0_1] = 1 + lam[es_0_1] * rho
    tau[es_1_0] = 1 + mu[es_1_0] * rho
    tau[es_1_1] = 1 - rho
    tau = np.clip(tau, 1e-8, None)  # evita log(negativo) si rho se saliera de su rango válido
    return float((np.log(tau) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu)).sum())


def ajustar_rho_dixon_coles(modelos: dict, X: pd.DataFrame, goles_local: pd.Series, goles_visitante: pd.Series) -> float:
    '''Busca por rejilla fina (rho en [-0.3, 0.3], paso 0.005) el valor que
    maximiza la log-verosimilitud de los marcadores realmente observados. Un
    Poisson bivariante independiente infravalora sistemáticamente los
    marcadores bajos (0-0, 1-0, 0-1, 1-1) frente a la correlación negativa
    leve que de verdad existe en el fútbol -- Dixon & Coles (1997) corrigen
    justo esas 4 celdas con este factor.
    '''
    lam = np.clip(modelos["local"].predict(X), 0.01, None)
    mu = np.clip(modelos["visitante"].predict(X), 0.01, None)
    x, y = goles_local.to_numpy(), goles_visitante.to_numpy()

    candidatos_rho = np.linspace(-0.3, 0.3, 121)
    log_verosimilitudes = [log_verosimilitud_dixon_coles(r, lam, mu, x, y) for r in candidatos_rho]
    return float(candidatos_rho[np.argmax(log_verosimilitudes)])


RHO_DIXON_COLES = ajustar_rho_dixon_coles(modelos_glm, X_train_inicial, df_train_inicial["goles_local"], df_train_inicial["goles_visitante"])
print(f"Corrección de Dixon-Coles: rho = {RHO_DIXON_COLES:.4f} (calibrado una vez sobre el entrenamiento pre-Mundial; "
      f"no se recalibra en el walk-forward, 13-32 partidos nuevos no lo van a mover sobre {len(df_train_inicial):,} históricos).")


def probabilidades_1x2_desde_lambdas(lam_local: np.ndarray, lam_visitante: np.ndarray,
                                       max_goles: int = 10) -> np.ndarray:
    '''Convierte vectores de (lambda_local, lambda_visitante) en probabilidades
    [P(local), P(empate), P(visitante)] recorriendo la rejilla conjunta de
    Poisson para cada partido, con la corrección de Dixon-Coles aplicada a los
    4 marcadores bajos -- la suma exacta de masa de probabilidad en cada
    región del marcador, no una aproximación cerrada.
    '''
    goles = np.arange(max_goles + 1)
    salida = np.zeros((len(lam_local), 3))
    for i, (la, lv) in enumerate(zip(lam_local, lam_visitante)):
        rejilla = np.outer(poisson.pmf(goles, la), poisson.pmf(goles, lv))
        rejilla[0, 0] *= 1 - la * lv * RHO_DIXON_COLES
        rejilla[0, 1] *= 1 + la * RHO_DIXON_COLES
        rejilla[1, 0] *= 1 + lv * RHO_DIXON_COLES
        rejilla[1, 1] *= 1 - RHO_DIXON_COLES
        rejilla = np.clip(rejilla, 0, None)
        salida[i] = [np.tril(rejilla, -1).sum(), np.trace(rejilla), np.triu(rejilla, 1).sum()]
    return salida / salida.sum(axis=1, keepdims=True)


def evaluar_modelo_goles(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame, nombre: str) -> dict:
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)

    rmse = np.sqrt(mean_squared_error(
        pd.concat([df_partidos["goles_local"], df_partidos["goles_visitante"]]),
        np.concatenate([pred_local, pred_visitante]),
    ))

    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante)
    y_real = df_partidos["resultado_1x2"].map({"LOCAL": 0, "EMPATE": 1, "VISITANTE": 2}).values
    logloss = log_loss(y_real, probs, labels=[0, 1, 2])

    print(f"[{nombre}] RMSE goles: {rmse:.4f}  |  LogLoss 1X2: {logloss:.4f}")
    return {"rmse": rmse, "logloss": logloss}


print("--- Fase de grupos ya jugada ---")
metricas_glm = evaluar_modelo_goles(modelos_glm, X_test_grupos, df_test_grupos, "Poisson GLM")
metricas_lgbm_base = evaluar_modelo_goles(modelos_lgbm_base, X_test_grupos, df_test_grupos, "LightGBM (sin ajustar)")
metricas_lgbm_optuna = evaluar_modelo_goles(modelos_lgbm_optuna, X_test_grupos, df_test_grupos, "LightGBM (Optuna)")
metricas_xgb_base = evaluar_modelo_goles(modelos_xgb_base, X_test_grupos, df_test_grupos, "XGBoost (sin ajustar)")
metricas_xgb_optuna = evaluar_modelo_goles(modelos_xgb_optuna, X_test_grupos, df_test_grupos, "XGBoost (Optuna)")

# Antes solo se comparaba GLM vs LightGBM, con "GLM gana" apoyado en un único
# torneo (2026). El backtesting del Notebook 3 contra 2010/2014/2018/2022
# mostro que XGBoost gana en RMSE y LogLoss en 3 de esos 4 Mundiales -- se
# incluye aqui como tercer candidato para que el propio walk-forward pueda
# elegirlo si tambien gana en 2026.
candidatos = {
    "glm": (modelos_glm, metricas_glm),
    "lgbm_optuna": (modelos_lgbm_optuna, metricas_lgbm_optuna),
    "xgboost": (modelos_xgb_optuna, metricas_xgb_optuna),
}
ganador_solo_2026 = min(candidatos, key=lambda k: candidatos[k][1]["rmse"])
print(f"Si decidiera solo el RMSE de este Mundial: {ganador_solo_2026} "
      f"(RMSE {candidatos[ganador_solo_2026][1]['rmse']:.4f}) -- pero 2026 es un solo torneo.")

# NO se usa `ganador_solo_2026`: el Notebook 3 (backtesting contra 2010, 2014,
# 2018 y 2022, entrenando cada vez solo con datos previos a cada Mundial)
# muestra que XGBoost gana en RMSE Y LogLoss en 3 de esos 4 torneos, con el
# mejor promedio de los 5 disponibles -- GLM solo gana en 2026, precisamente
# el unico que este notebook ve por si solo. Decidir con un solo holdout es
# exactamente el problema que el backtesting expuso (ver
# results/backtesting_resumen.csv), asi que aqui se fija el ganador segun esa
# evidencia mas amplia, no segun quien quede primero en este Mundial concreto.
nombre_ganador = "xgboost"
modelos_actuales = candidatos[nombre_ganador][0]
print(f"Modelo seleccionado para el walk-forward: {nombre_ganador} "
      f"(por el backtesting de 5 torneos, no por el RMSE de 2026 en solitario)")

Corrección de Dixon-Coles: rho = -0.0450 (calibrado una vez sobre el entrenamiento pre-Mundial; no se recalibra en el walk-forward, 13-32 partidos nuevos no lo van a mover sobre 32,286 históricos).
--- Fase de grupos ya jugada ---
[Poisson GLM] RMSE goles: 1.2700  |  LogLoss 1X2: 0.8979
[LightGBM (sin ajustar)] RMSE goles: 1.3054  |  LogLoss 1X2: 0.9131
[LightGBM (Optuna)] RMSE goles: 1.3069  |  LogLoss 1X2: 0.9154
[XGBoost (sin ajustar)] RMSE goles: 1.4187  |  LogLoss 1X2: 0.9438
[XGBoost (Optuna)] RMSE goles: 1.2959  |  LogLoss 1X2: 0.9035
Si decidiera solo el RMSE de este Mundial: glm (RMSE 1.2700) -- pero 2026 es un solo torneo.
Modelo seleccionado para el walk-forward: xgboost (por el backtesting de 5 torneos, no por el RMSE de 2026 en solitario)


### Predicciones de fase de grupos, partido a partido

La evaluación de arriba solo guarda el RMSE/LogLoss **agregado** de los 72 partidos de
grupos — útil para elegir familia, pero no deja ver qué predijo el modelo en un partido
concreto. Aquí se guarda esa vista partido a partido, con el modelo ganador entrenado
solo con datos pre-Mundial (el mismo que ya se evaluó arriba "en frío"). A diferencia de
eliminatoria, en fase de grupos **no se predice "quién avanza"**: la clasificación de un
grupo depende de la tabla completa (puntos, diferencia de goles...), no del resultado de
un único partido, así que aquí solo tiene sentido predecir goles y 1X2.

In [7]:
def marcador_moda_conjunta(lam_local: float, lam_visitante: float, max_goles: int = 10) -> str:
    '''Moda del marcador CONJUNTO (con Dixon-Coles), no floor(lambda) de cada
    lado por separado -- misma corrección que en 4.6/`resolver_eliminatoria`,
    duplicada aquí en vez de reutilizada porque `matriz_probabilidad_conjunta`
    todavía no existe en el notebook a esta altura (se define en 4.6).'''
    goles = np.arange(max_goles + 1)
    rejilla = np.outer(poisson.pmf(goles, lam_local), poisson.pmf(goles, lam_visitante))
    rejilla[0, 0] *= 1 - lam_local * lam_visitante * RHO_DIXON_COLES
    rejilla[0, 1] *= 1 + lam_local * RHO_DIXON_COLES
    rejilla[1, 0] *= 1 + lam_visitante * RHO_DIXON_COLES
    rejilla[1, 1] *= 1 - RHO_DIXON_COLES
    rejilla = np.clip(rejilla, 0, None)
    fila_moda, col_moda = np.unravel_index(np.argmax(rejilla), rejilla.shape)
    return f"{fila_moda}-{col_moda}"


def predecir_partidos_grupos(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame) -> pd.DataFrame:
    '''Predicción partido a partido de la fase de grupos. Sin `resolver_eliminatoria`
    a propósito: ese concepto de "quién avanza" solo aplica a un cruce directo de
    eliminatoria, no a un partido de grupos.'''
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)
    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante)
    etiquetas_1x2 = np.array(["LOCAL", "EMPATE", "VISITANTE"])

    filas = []
    for (_, fila), lam_l, lam_v, prob in zip(df_partidos.iterrows(), pred_local, pred_visitante, probs):
        filas.append({
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            "marcador_previsto": marcador_moda_conjunta(lam_l, lam_v),
            "prob_local": prob[0],
            "prob_empate": prob[1],
            "prob_visitante": prob[2],
            "resultado_1x2_previsto": etiquetas_1x2[np.argmax(prob)],
            "marcador_real": f"{int(fila['goles_local'])}-{int(fila['goles_visitante'])}",
            "resultado_1x2_real": fila["resultado_1x2"],
        })
    return pd.DataFrame(filas)


df_predicciones_grupos = predecir_partidos_grupos(candidatos[nombre_ganador][0], X_test_grupos, df_test_grupos)
ruta_predicciones_grupos = DIR_RESULTS / "predicciones_fase_grupos.csv"
df_predicciones_grupos.to_csv(ruta_predicciones_grupos, index=False)

aciertos_1x2 = (df_predicciones_grupos["resultado_1x2_previsto"] == df_predicciones_grupos["resultado_1x2_real"]).sum()
aciertos_marcador = (df_predicciones_grupos["marcador_previsto"] == df_predicciones_grupos["marcador_real"]).sum()
print(f"Guardado: {ruta_predicciones_grupos} ({len(df_predicciones_grupos)} partidos)")
print(f"Acierto 1X2: {aciertos_1x2}/{len(df_predicciones_grupos)}  |  Acierto marcador exacto: {aciertos_marcador}/{len(df_predicciones_grupos)}")
df_predicciones_grupos.head()

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/predicciones_fase_grupos.csv (72 partidos)
Acierto 1X2: 44/72  |  Acierto marcador exacto: 7/72


,fecha,equipo_local,equipo_visitante,lambda_local,lambda_visitante,marcador_previsto,prob_local,prob_empate,prob_visitante,resultado_1x2_previsto,marcador_real,resultado_1x2_real
0,2026-06-11,South Korea,Czech Republic,1.429206,1.000837,1-1,0.463435,0.278154,0.258412,LOCAL,2-1,LOCAL
1,2026-06-11,Mexico,South Africa,2.014094,0.702049,2-0,0.677601,0.206787,0.115612,LOCAL,2-0,LOCAL
2,2026-06-12,United States,Paraguay,1.374776,0.859428,1-0,0.483481,0.287633,0.228886,LOCAL,4-1,LOCAL
3,2026-06-12,Canada,Bosnia and Herzegovina,2.023958,0.786156,2-0,0.658422,0.210157,0.131421,LOCAL,1-1,EMPATE
4,2026-06-13,Haiti,Scotland,0.865148,1.534820,0-1,0.206560,0.268432,0.525008,VISITANTE,0-1,VISITANTE


### Checkpoint: guardar un snapshot del modelo por etapa, no solo el último

Hasta ahora `models/` solo guardaba el estado **más reciente**, sobreescrito cada vez
— no había forma de recuperar qué creía el modelo antes de la fase de grupos, o antes
de dieciseisavos, una vez reentrenado con la ronda siguiente. `guardar_checkpoint`
escribe un snapshot con nombre (`models/checkpoints/<etapa>/`) además de refrescar
`models/` con el estado actual, para poder comparar más adelante "qué predecía el
modelo en cada momento del torneo" — y para que la sección de Montecarlo (más abajo, en este mismo notebook) pueda reutilizar el
`pre_mundial` en vez de reentrenarlo desde cero.

In [8]:
def guardar_checkpoint(modelos: dict, nombre_etapa: str, familia: str, historico: pd.DataFrame,
                        metadatos_extra: dict | None = None) -> None:
    """Persiste el modelo dos veces: en `models/checkpoints/<nombre_etapa>/`
    (snapshot fijo de esa etapa, no se vuelve a tocar) y en `models/` (el
    estado "actual", que cada llamada posterior sí sobreescribe)."""
    metadatos = {
        "etapa": nombre_etapa,
        "familia": familia,
        "features": FEATURES_MODELO,
        "fecha_entrenamiento_utc": datetime.now(timezone.utc).isoformat(),
        "n_partidos_entrenamiento": len(historico),
        "ultima_fecha_entrenamiento": str(historico["fecha"].max().date()),
        **(metadatos_extra or {}),
    }
    for destino in (DIR_MODELOS / "checkpoints" / nombre_etapa, DIR_MODELOS):
        destino.mkdir(parents=True, exist_ok=True)
        joblib.dump(modelos["local"], destino / "modelo_goles_local.joblib")
        joblib.dump(modelos["visitante"], destino / "modelo_goles_visitante.joblib")
        with open(destino / "metadata.json", "w") as f:
            json.dump(metadatos, f, indent=2, default=str)
    print(f"Checkpoint '{nombre_etapa}' guardado ({len(historico):,} partidos, hasta "
          f"{metadatos['ultima_fecha_entrenamiento']}).")


guardar_checkpoint(
    candidatos[nombre_ganador][0], "pre_mundial", nombre_ganador, df_train_inicial,
    {"metricas_fase_grupos": {
        "glm": metricas_glm, "lgbm_base": metricas_lgbm_base, "lgbm_optuna": metricas_lgbm_optuna,
        "xgb_base": metricas_xgb_base, "xgb_optuna": metricas_xgb_optuna,
    }},
)

Checkpoint 'pre_mundial' guardado (32,286 partidos, hasta 2026-06-10).


## 4.6 Matriz de probabilidad conjunta y resolución de eliminatoria

Quién avanza se decide comparando **siempre** la probabilidad acumulada de cada lado en
la matriz de probabilidad conjunta (recorriendo todos los marcadores posibles de 0 a
`MAX_GOLES_MATRIZ`) — nunca comparando los marcadores más probables de cada lambda por
separado. Esa comparación puntual es más frágil de lo que parece: la moda de una
Poisson(λ) por sí sola no es lo mismo que "quién tiene más probabilidad de ganar el
partido", así que apoyarse en ella (con `round()` o incluso con `floor()`, que sí es la
moda correcta) para decidir el ganador seguiría siendo una aproximación innecesaria
cuando ya se dispone de la propia matriz de probabilidad.

In [9]:
def matriz_probabilidad_conjunta(lam_a: float, lam_b: float, max_goles: int = MAX_GOLES_MATRIZ) -> pd.DataFrame:
    '''Rejilla P(goles_A=i, goles_B=j) = Poisson(i;lam_a) * Poisson(j;lam_b),
    con la corrección de Dixon-Coles en los 4 marcadores bajos (mismo rho que
    en 4.5, calibrado una sola vez sobre el entrenamiento pre-Mundial).'''
    goles = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(goles, lam_a), poisson.pmf(goles, lam_b))
    matriz[0, 0] *= 1 - lam_a * lam_b * RHO_DIXON_COLES
    matriz[0, 1] *= 1 + lam_a * RHO_DIXON_COLES
    matriz[1, 0] *= 1 + lam_b * RHO_DIXON_COLES
    matriz[1, 1] *= 1 - RHO_DIXON_COLES
    matriz = np.clip(matriz, 0, None)
    matriz = matriz / matriz.sum()  # la corrección no preserva exactamente la masa total
    return pd.DataFrame(matriz, index=[f"A={i}" for i in goles], columns=[f"B={j}" for j in goles])


def resolver_eliminatoria(equipo_a: str, equipo_b: str, lam_a: float, lam_b: float) -> dict:
    '''Decide quién avanza comparando SIEMPRE la probabilidad acumulada de cada
    lado en la matriz de probabilidad conjunta -- nunca un marcador puntual de
    cada lambda por separado (ver 4.6).

    `marcador_previsto` es el argmax de esa MISMA matriz conjunta -- no
    floor(lambda) de cada lado por separado. Con Dixon-Coles (rho != 0) ambas
    cosas pueden diferir: floor/floor es la moda de cada Poisson marginal
    ASUMIENDO independencia, pero la corrección introduce correlación
    negativa leve justo en los marcadores bajos, así que la moda del
    marcador CONJUNTO ya no tiene por qué coincidir con la combinación de
    las dos modas marginales (ej. lam_a=0.99, lam_b=1.78 con rho=-0.045: el
    argmax conjunto es 1-1, no 0-1). Tampoco decide quién gana; eso sigue
    siendo la probabilidad acumulada de arriba.
    '''
    matriz = matriz_probabilidad_conjunta(lam_a, lam_b).values
    prob_gana_a = np.tril(matriz, -1).sum()
    prob_gana_b = np.triu(matriz, 1).sum()
    ganador = equipo_a if prob_gana_a > prob_gana_b else equipo_b
    fila_moda, col_moda = np.unravel_index(np.argmax(matriz), matriz.shape)

    return {"equipo_a": equipo_a, "equipo_b": equipo_b,
            "marcador_previsto": f"{fila_moda}-{col_moda}",
            "ganador": ganador, "metodo": f"matriz_conjunta (P(A)={prob_gana_a:.3f} vs P(B)={prob_gana_b:.3f})"}


# Ejemplo ilustrativo: un cruce parejo.
ejemplo = resolver_eliminatoria("Equipo A", "Equipo B", lam_a=1.35, lam_b=1.28)
print(ejemplo)
matriz_probabilidad_conjunta(1.35, 1.28).round(3)

{'equipo_a': 'Equipo A', 'equipo_b': 'Equipo B', 'marcador_previsto': '1-1', 'ganador': 'Equipo A', 'metodo': 'matriz_conjunta (P(A)=0.380 vs P(B)=0.347)'}


,B=0,B=1,B=2,B=3,B=4,B=5,B=6,B=7,B=8
A=0,0.078,0.087,0.059,0.025,0.008,0.002,0.000,0.0,0.0
A=1,0.092,0.130,0.080,0.034,0.011,0.003,0.001,0.0,0.0
A=2,0.066,0.084,0.054,0.023,0.007,0.002,0.000,0.0,0.0
A=3,0.030,0.038,0.024,0.010,0.003,0.001,0.000,0.0,0.0
A=4,0.010,0.013,0.008,0.003,0.001,0.000,0.000,0.0,0.0
A=5,0.003,0.003,0.002,0.001,0.000,0.000,0.000,0.0,0.0
A=6,0.001,0.001,0.000,0.000,0.000,0.000,0.000,0.0,0.0
A=7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0
A=8,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0


## 4.7 Bucle de walk-forward: eliminatorias con reentrenamiento continuo

Las rondas eliminatorias se agrupan por fecha (un salto de ≥2 días sin partidos
señala el cambio de ronda — no hace falta ninguna estructura de bracket externa,
solo el propio calendario real de partidos). Para cada ronda:

1. Se evalúa con el modelo tal como quedó **entrenado hasta ANTES** de esa ronda
   (nunca con datos que ya la incluyan — la misma disciplina de holdout de todo el
   proyecto).
2. Cada partido empatado a 90' se resuelve con la matriz de probabilidad conjunta.
3. Solo DESPUÉS de evaluar, los resultados reales de esa ronda se incorporan al
   histórico de entrenamiento — reentrenamiento completo (no incremental: con este
   volumen de datos el ajuste tarda segundos, y evita mantener a mano el estado
   incremental de las ventanas móviles) — antes de pasar a la siguiente ronda.

In [10]:
def agrupar_en_rondas(df_eliminatoria: pd.DataFrame, salto_dias: int = 2) -> list[pd.DataFrame]:
    '''Agrupa partidos de eliminatoria en rondas por proximidad de fecha.'''
    jugados = df_eliminatoria[df_eliminatoria["jugado"]].sort_values("fecha")
    if jugados.empty:
        return []
    salto = jugados["fecha"].diff().dt.days.fillna(0) > salto_dias
    jugados = jugados.assign(_ronda=salto.cumsum())
    return [grupo for _, grupo in jugados.groupby("_ronda")]


def reentrenar_modelos(df_historico_actualizado: pd.DataFrame, familia: str) -> dict:
    '''Reentrena desde cero (no incremental) la familia de modelo indicada
    ("glm", "lgbm_optuna" o "xgboost") con TODO el histórico ya actualizado.
    Los hiperparámetros de cada familia de árbol/boosting quedan fijos en los
    que ganó Optuna sobre el entrenamiento pre-Mundial (4.4) -- no se
    re-tunean en cada ronda del walk-forward, 13-32 partidos nuevos no lo van
    a mover frente a decenas de miles de históricos.
    '''
    X = df_historico_actualizado[FEATURES_MODELO]
    if familia == "glm":
        return {
            "local": entrenar_poisson_glm(X, df_historico_actualizado["goles_local"]),
            "visitante": entrenar_poisson_glm(X, df_historico_actualizado["goles_visitante"]),
        }
    if familia == "xgboost":
        return {
            "local": entrenar_xgboost_poisson(X, df_historico_actualizado["goles_local"], estudio_local_xgb.best_params),
            "visitante": entrenar_xgboost_poisson(X, df_historico_actualizado["goles_visitante"], estudio_visitante_xgb.best_params),
        }
    return {
        "local": entrenar_lightgbm_poisson(X, df_historico_actualizado["goles_local"], estudio_local.best_params),
        "visitante": entrenar_lightgbm_poisson(X, df_historico_actualizado["goles_visitante"], estudio_visitante.best_params),
    }


rondas = agrupar_en_rondas(df_eliminatoria)
print(f"Rondas de eliminatoria detectadas en el histórico (ya jugadas): {len(rondas)}")
for i, ronda in enumerate(rondas, start=1):
    print(f"  Ronda {i}: {ronda['fecha'].min().date()} -> {ronda['fecha'].max().date()}, {len(ronda)} partidos")

Rondas de eliminatoria detectadas en el histórico (ya jugadas): 1
  Ronda 1: 2026-06-28 -> 2026-07-03, 16 partidos


In [11]:
def probabilidades_partido(lam_local: float, lam_visitante: float) -> tuple[float, float, float]:
    '''Wrapper de una sola fila sobre `probabilidades_1x2_desde_lambdas`, para
    no repetir la lógica de la rejilla al construir el registro de cada
    predicción dentro del bucle.'''
    probs = probabilidades_1x2_desde_lambdas(np.array([lam_local]), np.array([lam_visitante]))
    return tuple(probs[0])


# Nombre legible de la etapa según el nº de partidos de la ronda (bracket de 32
# equipos: dieciseisavos=16, octavos=8, cuartos=4, semis=2, final=1); si no
# encaja (p.ej. una ronda incompleta, como ahora mismo) se usa "ronda_i".
ETAPAS_POR_TAMANO = {16: "dieciseisavos", 8: "octavos", 4: "cuartos", 2: "semis", 1: "final"}

historico_acumulado = pd.concat([df_train_inicial, df_test_grupos], ignore_index=True)
modelos_actuales = reentrenar_modelos(historico_acumulado, nombre_ganador)
print(f"Fase de grupos incorporada al entrenamiento ({len(historico_acumulado):,} partidos acumulados).")
guardar_checkpoint(modelos_actuales, "post_grupos", nombre_ganador, historico_acumulado)

resumen_walkforward = []
predicciones_partidos = []  # una fila por partido: predicción + resultado real, para poder auditar el modelo partido a partido

for i, ronda in enumerate(rondas, start=1):
    print(f"\n=== Ronda {i} ({len(ronda)} partidos, {ronda['fecha'].min().date()}) ===")

    # 1. Evaluar ESTA ronda con el modelo tal como estaba ANTES de conocerla.
    X_ronda = ronda[FEATURES_MODELO]
    metricas_ronda = evaluar_modelo_goles(modelos_actuales, X_ronda, ronda, f"ronda {i} (modelo pre-ronda)")

    # 2. Resolver cada partido, con matriz conjunta si el marcador previsto es empate.
    pred_local = np.clip(modelos_actuales["local"].predict(X_ronda), 0.01, None)
    pred_visitante = np.clip(modelos_actuales["visitante"].predict(X_ronda), 0.01, None)
    for (_, fila), lam_l, lam_v in zip(ronda.iterrows(), pred_local, pred_visitante):
        resultado = resolver_eliminatoria(fila["equipo_local"], fila["equipo_visitante"], lam_l, lam_v)
        prob_local, prob_empate, prob_visitante = probabilidades_partido(lam_l, lam_v)
        print(f"  {resultado['equipo_a']} vs {resultado['equipo_b']}: "
              f"previsto {resultado['marcador_previsto']}, avanza {resultado['ganador']} ({resultado['metodo']})")
        # Nota: "marcador_real_90min" es el resultado reglamentario (lo único que
        # modela el sistema); en un cruce que se decidiera en prórroga/penaltis
        # esto NO tiene por qué coincidir con quién avanzó realmente al torneo.
        predicciones_partidos.append({
            "ronda": i,
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            "marcador_previsto": resultado["marcador_previsto"],
            "prob_local": prob_local,
            "prob_empate": prob_empate,
            "prob_visitante": prob_visitante,
            "avanza_previsto": resultado["ganador"],
            "metodo_desempate": resultado["metodo"],
            "marcador_real_90min": f"{int(fila['goles_local'])}-{int(fila['goles_visitante'])}",
            "resultado_1x2_real": fila["resultado_1x2"],
        })

    resumen_walkforward.append({"ronda": i, "n_partidos": len(ronda), **metricas_ronda})

    # 3. Solo AHORA se incorporan los resultados reales de esta ronda, y se
    #    reentrena de cara a la siguiente — el "rolling update" del pipeline.
    historico_acumulado = pd.concat([historico_acumulado, ronda], ignore_index=True)
    modelos_actuales = reentrenar_modelos(historico_acumulado, nombre_ganador)
    nombre_etapa = f"post_{ETAPAS_POR_TAMANO.get(len(ronda), f'ronda_{i}')}"
    guardar_checkpoint(modelos_actuales, nombre_etapa, nombre_ganador, historico_acumulado, {"metricas_ronda": metricas_ronda})
    print(f"  Reentrenado con {len(historico_acumulado):,} partidos acumulados.")

if resumen_walkforward:
    print("\n=== Evolución de las métricas ronda a ronda ===")
    print(pd.DataFrame(resumen_walkforward).round(4).to_string(index=False))
else:
    print("Sin rondas de eliminatoria jugadas todavía en el histórico disponible.")

Fase de grupos incorporada al entrenamiento (32,358 partidos acumulados).
Checkpoint 'post_grupos' guardado (32,358 partidos, hasta 2026-06-27).

=== Ronda 1 (16 partidos, 2026-06-28) ===
[ronda 1 (modelo pre-ronda)] RMSE goles: 0.7139  |  LogLoss 1X2: 0.7563
  South Africa vs Canada: previsto 1-1, avanza South Africa (matriz_conjunta (P(A)=0.380 vs P(B)=0.340))
  Netherlands vs Morocco: previsto 1-1, avanza Netherlands (matriz_conjunta (P(A)=0.492 vs P(B)=0.233))
  Germany vs Paraguay: previsto 1-0, avanza Germany (matriz_conjunta (P(A)=0.570 vs P(B)=0.177))
  Brazil vs Japan: previsto 1-0, avanza Brazil (matriz_conjunta (P(A)=0.576 vs P(B)=0.173))
  France vs Sweden: previsto 1-0, avanza France (matriz_conjunta (P(A)=0.639 vs P(B)=0.131))
  Ivory Coast vs Norway: previsto 1-1, avanza Norway (matriz_conjunta (P(A)=0.303 vs P(B)=0.396))
  Mexico vs Ecuador: previsto 1-0, avanza Mexico (matriz_conjunta (P(A)=0.577 vs P(B)=0.165))
  England vs DR Congo: previsto 2-0, avanza England (matr

Checkpoint 'post_dieciseisavos' guardado (32,374 partidos, hasta 2026-07-03).
  Reentrenado con 32,374 partidos acumulados.

=== Evolución de las métricas ronda a ronda ===
 ronda  n_partidos   rmse  logloss
     1          16 0.7139   0.7563


## 4.8 Persistencia: modelo final y predicciones

El resto del notebook vivía solo en memoria de la sesión — ni el modelo ganador
ni las predicciones partido a partido quedaban en disco, así que reabrir el
notebook obligaba a reentrenar todo desde cero para volver a consultarlas. Dos
artefactos, ambos derivados y por tanto en `.gitignore`:

- `data/processed/predicciones_eliminatoria.csv`: una fila por partido de
  eliminatoria con la predicción completa (lambdas, marcador previsto,
  probabilidades 1X2, método de desempate) y el resultado real al lado, para
  poder auditar el modelo partido a partido sin releer los prints del bucle.
- `models/`: el modelo ganador (`nombre_ganador`), reentrenado con **todo** el
  histórico acumulado hasta la última ronda conocida — es el que se usaría
  para predecir la siguiente ronda todavía no jugada — junto a un
  `metadata.json` con de qué familia es, qué features espera, cuándo se
  entrenó y con qué métricas.


In [12]:
df_predicciones = pd.DataFrame(predicciones_partidos)
ruta_predicciones = DIR_RESULTS / "predicciones_eliminatoria.csv"
df_predicciones.to_csv(ruta_predicciones, index=False)
print(f"Predicciones de eliminatoria guardadas: {ruta_predicciones} ({len(df_predicciones)} partidos)")
df_predicciones


Predicciones de eliminatoria guardadas: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/predicciones_eliminatoria.csv (16 partidos)


,ronda,fecha,equipo_local,equipo_visitante,lambda_local,lambda_visitante,marcador_previsto,prob_local,prob_empate,prob_visitante,avanza_previsto,metodo_desempate,marcador_real_90min,resultado_1x2_real
0,1,2026-06-28,South Africa,Canada,1.301355,1.215995,1-1,0.380204,0.280143,0.339653,South Africa,matriz_conjunta (P(A)=0.380 vs P(B)=0.340),0-1,VISITANTE
1,1,2026-06-29,Netherlands,Morocco,1.478523,0.936316,1-1,0.492331,0.274269,0.233399,Netherlands,matriz_conjunta (P(A)=0.492 vs P(B)=0.233),1-1,EMPATE
2,1,2026-06-29,Germany,Paraguay,1.663397,0.814826,1-0,0.570382,0.252682,0.176936,Germany,matriz_conjunta (P(A)=0.570 vs P(B)=0.177),1-1,EMPATE
3,1,2026-06-29,Brazil,Japan,1.675833,0.804028,1-0,0.576216,0.250977,0.172807,Brazil,matriz_conjunta (P(A)=0.576 vs P(B)=0.173),2-1,LOCAL
4,1,2026-06-30,France,Sweden,1.818739,0.690380,1-0,0.639354,0.229566,0.131080,France,matriz_conjunta (P(A)=0.639 vs P(B)=0.131),3-0,LOCAL
5,1,2026-06-30,Ivory Coast,Norway,1.011871,1.197435,1-1,0.302998,0.300905,0.396096,Norway,matriz_conjunta (P(A)=0.303 vs P(B)=0.396),1-2,VISITANTE
6,1,2026-06-30,Mexico,Ecuador,1.609417,0.739847,1-0,0.576768,0.257963,0.165269,Mexico,matriz_conjunta (P(A)=0.577 vs P(B)=0.165),2-0,LOCAL
7,1,2026-07-01,England,DR Congo,2.330036,0.731455,2-0,0.728780,0.174968,0.096252,England,matriz_conjunta (P(A)=0.729 vs P(B)=0.096),2-1,LOCAL
8,1,2026-07-01,United States,Bosnia and Herzegovina,1.771409,0.663499,1-0,0.635879,0.234349,0.129772,United States,matriz_conjunta (P(A)=0.636 vs P(B)=0.130),2-0,LOCAL
9,1,2026-07-01,Belgium,Senegal,2.092494,0.980083,2-0,0.625318,0.211130,0.163551,Belgium,matriz_conjunta (P(A)=0.625 vs P(B)=0.164),2-2,EMPATE


In [13]:
import json
from datetime import datetime, timezone

import joblib

DIR_MODELOS = DIR_RAIZ / "models"
DIR_MODELOS.mkdir(exist_ok=True)

joblib.dump(modelos_actuales["local"], DIR_MODELOS / "modelo_goles_local.joblib")
joblib.dump(modelos_actuales["visitante"], DIR_MODELOS / "modelo_goles_visitante.joblib")

metadatos_modelo = {
    "familia": nombre_ganador,
    "features": FEATURES_MODELO,
    "fecha_entrenamiento_utc": datetime.now(timezone.utc).isoformat(),
    "n_partidos_entrenamiento": len(historico_acumulado),
    "ultima_fecha_entrenamiento": str(historico_acumulado["fecha"].max().date()),
    "metricas_fase_grupos": {
        "glm": metricas_glm,
        "lgbm_base": metricas_lgbm_base,
        "lgbm_optuna": metricas_lgbm_optuna,
    },
    "metricas_walkforward_por_ronda": resumen_walkforward,
    "hiperparametros_lgbm_optuna": {
        "local": estudio_local.best_params,
        "visitante": estudio_visitante.best_params,
    } if nombre_ganador == "lgbm_optuna" else None,
}
with open(DIR_MODELOS / "metadata.json", "w") as f:
    json.dump(metadatos_modelo, f, indent=2, default=str)

print(f"Modelo '{nombre_ganador}' persistido en {DIR_MODELOS} (entrenado con {len(historico_acumulado):,} partidos, "
      f"hasta {metadatos_modelo['ultima_fecha_entrenamiento']}).")


Modelo 'xgboost' persistido en /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/models (entrenado con 32,374 partidos, hasta 2026-07-03).


## 4.9 Predicción de los próximos partidos (todavía sin jugar)

Todo lo anterior evalúa partidos que **ya se jugaron** (para poder comparar contra el
resultado real). Esta sección hace lo que de verdad importa de cara a lo que queda de
torneo: usar el modelo actual (`modelos_actuales`, ya reentrenado con todo lo conocido
hasta la última ronda) para predecir los cruces de eliminatoria **pendientes** —
incluidos los de octavos que ya se conocen aunque todavía no se hayan disputado
(algunos octavos siguen bloqueados por un dieciseisavo de hoy sin resolver, p. ej.
`"W83"`/`"W88"` en el calendario original; esas filas ni siquiera llegan a
`partidos_features.csv` hasta que el cruce anterior se juega, así que se filtran por
seguridad si alguna vez aparecieran).

In [14]:
partidos_pendientes = df_eliminatoria[~df_eliminatoria["jugado"]].copy()

# Defensivo: un cruce de bracket sin resolver todavía (p.ej. "W83") no tiene
# historial de Elo/forma propio -- no hay nada que predecir para él.
patron_placeholder = r"^[WL]\d+$"
partidos_pendientes = partidos_pendientes[
    ~partidos_pendientes["equipo_local"].str.match(patron_placeholder)
    & ~partidos_pendientes["equipo_visitante"].str.match(patron_placeholder)
].sort_values("fecha")

predicciones_pendientes = []
if not partidos_pendientes.empty:
    X_pendientes = partidos_pendientes[FEATURES_MODELO]
    pred_local = np.clip(modelos_actuales["local"].predict(X_pendientes), 0.01, None)
    pred_visitante = np.clip(modelos_actuales["visitante"].predict(X_pendientes), 0.01, None)

    print(f"=== Predicción con el modelo actual ({nombre_ganador}, entrenado hasta "
          f"{historico_acumulado['fecha'].max().date()}) ===")
    for (_, fila), lam_l, lam_v in zip(partidos_pendientes.iterrows(), pred_local, pred_visitante):
        resultado = resolver_eliminatoria(fila["equipo_local"], fila["equipo_visitante"], lam_l, lam_v)
        prob_local, prob_empate, prob_visitante = probabilidades_partido(lam_l, lam_v)
        print(f"  [{fila['fecha'].date()}] {resultado['equipo_a']} vs {resultado['equipo_b']}: "
              f"previsto {resultado['marcador_previsto']}, avanza {resultado['ganador']} ({resultado['metodo']})")
        predicciones_pendientes.append({
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            "marcador_previsto": resultado["marcador_previsto"],
            "prob_local": prob_local,
            "prob_empate": prob_empate,
            "prob_visitante": prob_visitante,
            "avanza_previsto": resultado["ganador"],
            "metodo_desempate": resultado["metodo"],
        })
else:
    print("No hay partidos de eliminatoria pendientes con ambos equipos ya conocidos.")

df_predicciones_pendientes = pd.DataFrame(predicciones_pendientes)
ruta_predicciones_pendientes = DIR_RESULTS / "predicciones_proximos_partidos.csv"
df_predicciones_pendientes.to_csv(ruta_predicciones_pendientes, index=False)
print(f"\nGuardado: {ruta_predicciones_pendientes} ({len(df_predicciones_pendientes)} partidos)")
df_predicciones_pendientes

=== Predicción con el modelo actual (xgboost, entrenado hasta 2026-07-03) ===
  [2026-07-04] Paraguay vs France: previsto 0-2, avanza France (matriz_conjunta (P(A)=0.085 vs P(B)=0.758))
  [2026-07-04] Canada vs Morocco: previsto 1-1, avanza Morocco (matriz_conjunta (P(A)=0.240 vs P(B)=0.498))
  [2026-07-05] Mexico vs England: previsto 1-1, avanza England (matriz_conjunta (P(A)=0.313 vs P(B)=0.391))
  [2026-07-05] Brazil vs Norway: previsto 1-1, avanza Brazil (matriz_conjunta (P(A)=0.422 vs P(B)=0.282))
  [2026-07-06] Portugal vs Spain: previsto 1-1, avanza Spain (matriz_conjunta (P(A)=0.236 vs P(B)=0.490))
  [2026-07-06] Argentina vs Egypt: previsto 3-0, avanza Argentina (matriz_conjunta (P(A)=0.888 vs P(B)=0.026))
  [2026-07-06] United States vs Belgium: previsto 1-1, avanza Belgium (matriz_conjunta (P(A)=0.281 vs P(B)=0.434))
  [2026-07-06] Switzerland vs Colombia: previsto 1-1, avanza Colombia (matriz_conjunta (P(A)=0.278 vs P(B)=0.452))

Guardado: /Users/danielcanteragomez/portfoli

,fecha,equipo_local,equipo_visitante,lambda_local,lambda_visitante,marcador_previsto,prob_local,prob_empate,prob_visitante,avanza_previsto,metodo_desempate
0,2026-07-04,Paraguay,France,0.745068,2.532875,0-2,0.084893,0.156785,0.758323,France,matriz_conjunta (P(A)=0.085 vs P(B)=0.758)
1,2026-07-04,Canada,Morocco,1.025865,1.584121,1-1,0.240112,0.262061,0.497826,Morocco,matriz_conjunta (P(A)=0.240 vs P(B)=0.498)
2,2026-07-05,Mexico,England,1.063079,1.220379,1-1,0.313232,0.295676,0.391092,England,matriz_conjunta (P(A)=0.313 vs P(B)=0.391)
3,2026-07-05,Brazil,Norway,1.262880,0.982942,1-1,0.421646,0.295904,0.282450,Brazil,matriz_conjunta (P(A)=0.422 vs P(B)=0.282)
4,2026-07-06,Portugal,Spain,0.945283,1.478192,1-1,0.235807,0.274167,0.490027,Spain,matriz_conjunta (P(A)=0.236 vs P(B)=0.490)
5,2026-07-06,Argentina,Egypt,3.131304,0.435935,3-0,0.888531,0.085605,0.025864,Argentina,matriz_conjunta (P(A)=0.888 vs P(B)=0.026)
6,2026-07-06,United States,Belgium,1.037386,1.354020,1-1,0.281212,0.284426,0.434362,Belgium,matriz_conjunta (P(A)=0.281 vs P(B)=0.434)
7,2026-07-06,Switzerland,Colombia,1.108145,1.481240,1-1,0.277869,0.270167,0.451964,Colombia,matriz_conjunta (P(A)=0.278 vs P(B)=0.452)


## 4.10 Calibración de probabilidades

Todo lo anterior usa las probabilidades 1X2 para decidir marcadores, empates y avances
— pero nunca se ha comprobado si esas probabilidades están **bien calibradas**: de los
partidos donde el modelo dice "60% de probabilidad de victoria local", ¿gana el local
de verdad ~60% de las veces? Con los 85 partidos ya jugados (72 de grupos + 13 de
eliminatoria) hay señal suficiente para un diagrama de fiabilidad simplificado, aunque
con pocos bins — la muestra sigue siendo pequeña para sacar conclusiones fuertes.

In [15]:
def diagrama_calibracion(df: pd.DataFrame, columna_prob: str, etiqueta: str, n_bins: int = 4) -> pd.DataFrame:
    """Reliability diagram simplificado: agrupa los partidos por la probabilidad
    predicha de `etiqueta` (LOCAL/EMPATE/VISITANTE) en `n_bins` cuantiles y
    compara la media predicha con la frecuencia observada en cada bin -- si el
    modelo está bien calibrado, ambas deberían parecerse.
    """
    df = df.copy()
    n_bins_reales = min(n_bins, df[columna_prob].nunique())
    df["_bin"] = pd.qcut(df[columna_prob], q=n_bins_reales, duplicates="drop")
    return df.groupby("_bin", observed=True).agg(
        n_partidos=(columna_prob, "size"),
        prob_media_predicha=(columna_prob, "mean"),
        frecuencia_observada=("resultado_1x2_real", lambda s: (s == etiqueta).mean()),
    ).reset_index(drop=True)


columnas_comunes = ["prob_local", "prob_empate", "prob_visitante", "resultado_1x2_real"]
df_todas_predicciones = pd.concat([
    df_predicciones_grupos[columnas_comunes],
    df_predicciones[columnas_comunes],
], ignore_index=True)

print(f"Calibración sobre {len(df_todas_predicciones)} partidos ya jugados (fase de grupos + eliminatoria).")
for columna, etiqueta in [("prob_local", "LOCAL"), ("prob_visitante", "VISITANTE")]:
    print(f"\n--- P({etiqueta.lower()}) ---")
    print(diagrama_calibracion(df_todas_predicciones, columna, etiqueta).round(3).to_string(index=False))


Calibración sobre 88 partidos ya jugados (fase de grupos + eliminatoria).

--- P(local) ---
 n_partidos  prob_media_predicha  frecuencia_observada
         22                0.164                 0.182
         22                0.414                 0.500
         22                0.573                 0.455
         22                0.713                 0.818

--- P(visitante) ---
 n_partidos  prob_media_predicha  frecuencia_observada
         22                0.103                 0.000
         22                0.178                 0.091
         22                0.302                 0.182
         22                0.609                 0.682


## 4.11 Evaluación de clasificación 1X2: matriz de confusión, precisión, recall, F1

LogLoss (4.5) mide la calidad de las probabilidades, pero no dice directamente "si
tuvieras que apostar por un resultado, ¿cuántas veces aciertas, y en cuáles te
equivocas más?". Para eso hace falta tratar la predicción como lo que también es -- un
clasificador de 3 clases (LOCAL/EMPATE/VISITANTE, tomando la de mayor probabilidad) -- y
mirarlo con las herramientas estándar de clasificación:

- **Matriz de confusión**: no solo cuántos aciertos, sino **con qué se confunde** el
  modelo (¿predice LOCAL cuando en realidad es EMPATE, o VISITANTE?).
- **Precisión / recall / F1 por clase**: el fútbol tiene clases desequilibradas (hay
  bastantes menos empates que victorias locales), así que un accuracy global puede
  esconder que el modelo sea mucho peor prediciendo empates que victorias.
- **Accuracy balanceada** (media del recall de cada clase) y **Cohen's Kappa** (acuerdo
  con la realidad por encima de lo que daría el puro azar/desequilibrio de clases):
  ambas corrigen el mismo problema que el accuracy bruto puede maquillar.
- **Brier score multiclase**: el equivalente "en distancia euclídea" de LogLoss sobre
  las probabilidades -- menos castigador con un solo fallo de probabilidad extrema,
  buen complementario para no depender de una única métrica de calibración.

Todo comparado contra una **baseline ingenua** (predecir siempre la clase más frecuente
observada) -- sin esa comparación, un accuracy del 50% no dice nada por sí solo.

In [16]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    cohen_kappa_score, confusion_matrix,
)

ETIQUETAS_1X2 = ["LOCAL", "EMPATE", "VISITANTE"]


def evaluar_clasificacion_1x2(df: pd.DataFrame, nombre: str) -> dict:
    """Evalúa la predicción 1X2 (la clase de mayor probabilidad) como lo que
    también es: un clasificador de 3 clases. `df` debe traer prob_local,
    prob_empate, prob_visitante y resultado_1x2_real -- el mismo esquema que
    ya usan predicciones_fase_grupos.csv y predicciones_eliminatoria.csv.
    """
    y_real = df["resultado_1x2_real"].to_numpy()
    probs = df[["prob_local", "prob_empate", "prob_visitante"]].to_numpy()
    y_pred = np.array(ETIQUETAS_1X2)[probs.argmax(axis=1)]

    clase_mayoritaria = pd.Series(y_real).mode().iloc[0]
    acc_baseline = (y_real == clase_mayoritaria).mean()
    acc = accuracy_score(y_real, y_pred)
    acc_balanceada = balanced_accuracy_score(y_real, y_pred)
    kappa = cohen_kappa_score(y_real, y_pred)

    y_real_onehot = np.column_stack([(y_real == e).astype(int) for e in ETIQUETAS_1X2])
    brier = float(np.mean(np.sum((probs - y_real_onehot) ** 2, axis=1)))

    print(f"=== {nombre} ({len(df)} partidos) ===")
    print(f"Distribución real:     {pd.Series(y_real).value_counts().to_dict()}")
    print(f"Distribución prevista: {pd.Series(y_pred).value_counts().to_dict()}")
    print()
    print(f"Accuracy:            {acc:.3f}   (baseline 'siempre {clase_mayoritaria}': {acc_baseline:.3f})")
    print(f"Accuracy balanceada: {acc_balanceada:.3f}   (media del recall por clase)")
    print(f"Cohen's Kappa:       {kappa:+.3f}   (0 = igual que el azar/baseline, 1 = perfecto)")
    print(f"Brier score:         {brier:.4f}   (0 = perfecto, 2 = peor posible con 3 clases)")
    print()
    print("Matriz de confusión (filas = real, columnas = previsto):")
    matriz_conf = confusion_matrix(y_real, y_pred, labels=ETIQUETAS_1X2)
    print(pd.DataFrame(matriz_conf, index=[f"real_{e}" for e in ETIQUETAS_1X2],
                        columns=[f"prev_{e}" for e in ETIQUETAS_1X2]).to_string())
    print()
    print("Precisión / recall / F1 por clase:")
    print(classification_report(y_real, y_pred, labels=ETIQUETAS_1X2, target_names=ETIQUETAS_1X2, zero_division=0))

    return {"nombre": nombre, "n": len(df), "accuracy": acc, "accuracy_baseline": acc_baseline,
            "accuracy_balanceada": acc_balanceada, "kappa": kappa, "brier": brier}


columnas_comunes_1x2 = ["prob_local", "prob_empate", "prob_visitante", "resultado_1x2_real"]
df_todas_predicciones_1x2 = pd.concat([
    df_predicciones_grupos[columnas_comunes_1x2],
    df_predicciones[columnas_comunes_1x2],
], ignore_index=True)

resumen_clasificacion = []
resumen_clasificacion.append(evaluar_clasificacion_1x2(df_predicciones_grupos, "Fase de grupos"))
print()
resumen_clasificacion.append(evaluar_clasificacion_1x2(df_predicciones, "Eliminatoria (ronda 1)"))
print()
resumen_clasificacion.append(evaluar_clasificacion_1x2(df_todas_predicciones_1x2, "Combinado (grupos + eliminatoria)"))

pd.DataFrame(resumen_clasificacion).round(4)

=== Fase de grupos (72 partidos) ===
Distribución real:     {'LOCAL': 33, 'EMPATE': 20, 'VISITANTE': 19}
Distribución prevista: {'LOCAL': 47, 'VISITANTE': 25}

Accuracy:            0.611   (baseline 'siempre LOCAL': 0.458)
Accuracy balanceada: 0.564   (media del recall por clase)
Cohen's Kappa:       +0.362   (0 = igual que el azar/baseline, 1 = perfecto)
Brier score:         0.5306   (0 = perfecto, 2 = peor posible con 3 clases)

Matriz de confusión (filas = real, columnas = previsto):
                prev_LOCAL  prev_EMPATE  prev_VISITANTE
real_LOCAL              28            0               5
real_EMPATE             16            0               4
real_VISITANTE           3            0              16

Precisión / recall / F1 por clase:
              precision    recall  f1-score   support

       LOCAL       0.60      0.85      0.70        33
      EMPATE       0.00      0.00      0.00        20
   VISITANTE       0.64      0.84      0.73        19

    accuracy                  

,nombre,n,accuracy,accuracy_baseline,accuracy_balanceada,kappa,brier
0,Fase de grupos,72,0.6111,0.4583,0.5635,0.3616,0.5306
1,Eliminatoria (ronda 1),16,0.6875,0.6250,0.5000,0.2857,0.4323
2,Combinado (grupos + eliminatoria),88,0.6250,0.4886,0.5644,0.3623,0.5127


## 4.12 Evaluación del marcador previsto (una pregunta distinta de 4.11)

`marcador_previsto` (floor(lambda) de cada lado, por separado) y la clasificación 1X2
de 4.11 (probabilidad agregada de toda una región del marcador) responden preguntas
distintas y **pueden apuntar a ganadores diferentes para el mismo partido** -- el caso
Países Bajos-Marruecos: marcador previsto 1-1, pero P(local)=0.49 > P(empate)=0.26, así
que 4.11 lo clasifica como "LOCAL". Aquí se evalúa el marcador en sus propios términos
(acierto exacto, error medio en goles) y se mide cuántas veces discrepa del veredicto de
4.11, para dejar cuantificado ese fenómeno en vez de descubrirlo partido a partido.

In [17]:
def _goles_de_marcador(marcador: np.ndarray) -> np.ndarray:
    return np.array([tuple(map(int, m.split("-"))) for m in marcador])


def evaluar_marcador_previsto(df: pd.DataFrame, nombre: str) -> dict:
    goles_prev = _goles_de_marcador(df["marcador_previsto"].to_numpy())
    goles_real = _goles_de_marcador(df["marcador_real"].to_numpy())

    acierto_exacto = (df["marcador_previsto"].to_numpy() == df["marcador_real"].to_numpy())
    error_abs_local = np.abs(goles_prev[:, 0] - goles_real[:, 0])
    error_abs_visitante = np.abs(goles_prev[:, 1] - goles_real[:, 1])

    signo = lambda d: np.where(d > 0, "LOCAL", np.where(d < 0, "VISITANTE", "EMPATE"))
    ganador_por_marcador = signo(goles_prev[:, 0] - goles_prev[:, 1])
    ganador_real = signo(goles_real[:, 0] - goles_real[:, 1])
    acierto_ganador_marcador = (ganador_por_marcador == ganador_real)

    etiquetas_1x2 = np.array(["LOCAL", "EMPATE", "VISITANTE"])
    probs = df[["prob_local", "prob_empate", "prob_visitante"]].to_numpy()
    resultado_1x2_previsto = etiquetas_1x2[probs.argmax(axis=1)]
    discrepancia = (ganador_por_marcador != resultado_1x2_previsto)

    print(f"=== {nombre} ({len(df)} partidos) ===")
    print(f"Acierto de marcador exacto:                  {acierto_exacto.mean():.3f}  ({acierto_exacto.sum()}/{len(df)})")
    print(f"Error absoluto medio, goles local:            {error_abs_local.mean():.3f}")
    print(f"Error absoluto medio, goles visitante:        {error_abs_visitante.mean():.3f}")
    print(f"Acierto de ganador según el marcador:          {acierto_ganador_marcador.mean():.3f}")
    print(f"Discrepancia marcador vs. clasificación 1X2:  {discrepancia.mean():.3f}  "
          f"({discrepancia.sum()}/{len(df)} partidos tipo Países Bajos-Marruecos)")

    return {"nombre": nombre, "n": len(df), "acierto_marcador_exacto": acierto_exacto.mean(),
            "mae_goles_local": error_abs_local.mean(), "mae_goles_visitante": error_abs_visitante.mean(),
            "acierto_ganador_marcador": acierto_ganador_marcador.mean(),
            "discrepancia_vs_1x2": discrepancia.mean()}


df_grupos_marcador = df_predicciones_grupos[["marcador_previsto", "marcador_real", "prob_local", "prob_empate", "prob_visitante"]].copy()
df_elim_marcador = df_predicciones[["marcador_previsto", "prob_local", "prob_empate", "prob_visitante"]].copy()
df_elim_marcador["marcador_real"] = df_predicciones["marcador_real_90min"]
df_todas_marcador = pd.concat([df_grupos_marcador, df_elim_marcador], ignore_index=True)

resumen_marcador = []
resumen_marcador.append(evaluar_marcador_previsto(df_grupos_marcador, "Fase de grupos"))
print()
resumen_marcador.append(evaluar_marcador_previsto(df_elim_marcador, "Eliminatoria (ronda 1)"))
print()
resumen_marcador.append(evaluar_marcador_previsto(df_todas_marcador, "Combinado (grupos + eliminatoria)"))

pd.DataFrame(resumen_marcador).round(4)

=== Fase de grupos (72 partidos) ===
Acierto de marcador exacto:                  0.097  (7/72)
Error absoluto medio, goles local:            1.222
Error absoluto medio, goles visitante:        0.875
Acierto de ganador según el marcador:          0.583
Discrepancia marcador vs. clasificación 1X2:  0.292  (21/72 partidos tipo Países Bajos-Marruecos)

=== Eliminatoria (ronda 1) (16 partidos) ===
Acierto de marcador exacto:                  0.125  (2/16)
Error absoluto medio, goles local:            0.750
Error absoluto medio, goles visitante:        0.500
Acierto de ganador según el marcador:          0.688
Discrepancia marcador vs. clasificación 1X2:  0.312  (5/16 partidos tipo Países Bajos-Marruecos)

=== Combinado (grupos + eliminatoria) (88 partidos) ===
Acierto de marcador exacto:                  0.102  (9/88)
Error absoluto medio, goles local:            1.136
Error absoluto medio, goles visitante:        0.807
Acierto de ganador según el marcador:          0.602
Discrepancia marc

,nombre,n,acierto_marcador_exacto,mae_goles_local,mae_goles_visitante,acierto_ganador_marcador,discrepancia_vs_1x2
0,Fase de grupos,72,0.0972,1.2222,0.8750,0.5833,0.2917
1,Eliminatoria (ronda 1),16,0.1250,0.7500,0.5000,0.6875,0.3125
2,Combinado (grupos + eliminatoria),88,0.1023,1.1364,0.8068,0.6023,0.2955


## Resumen de la parte de predicción (4.1-4.12)

- Modelo de goles seleccionado (RMSE + LogLoss en fase de grupos): impreso en la
  sección 4.5 (`nombre_ganador`).
- El bucle de la sección 4.7 demuestra el ciclo completo de un sistema de predicción
  vivo: evaluar honestamente ANTES de conocer el resultado, e incorporar el dato real
  DESPUÉS — nunca al revés.
- La matriz de probabilidad conjunta (4.6) es la pieza que resuelve la ambigüedad de
  un empate a 90 minutos en una eliminatoria directa, sin necesitar simular
  prórroga/penaltis explícitamente: el equipo con más probabilidad acumulada en su
  mitad del marcador es el que se da por ganador.

## 4.13 Simulación Montecarlo: probabilidad de cada selección por ronda

Todo lo anterior (4.1-4.12) predice partido a partido, un marcador a la vez. Aquí se simula el
**cuadro completo, miles de veces**, para responder una pregunta distinta: "¿qué probabilidad
tiene España de llegar a semifinales?" -- no se puede leer eso de una única predicción
determinista, hace falta propagar la incertidumbre de cada cruce con sorteos de Poisson.

**`ETAPA_SIMULACION` decide el punto de vista**:
- `"pre_mundial"`: el modelo tal como estaba el día ANTES de empezar el Mundial, sin haber
  visto ni la fase de grupos -- útil para comparar "qué se esperaba" contra lo que pasó de
  verdad.
- `"actual"`: el checkpoint más reciente disponible (todo lo jugado hasta hoy ya incorporado),
  fijando como CONOCIDOS los partidos que ya se jugaron de verdad -- solo se simula lo que
  todavía está por decidir. Esto es lo que quieres si el Mundial ya lleva rondas jugadas y
  buscas probabilidades actualizadas.

Cambia la constante de abajo y vuelve a ejecutar desde aquí (4.13 en adelante) para pasar de
una vista a la otra -- no hace falta re-ejecutar 4.1-4.12.

In [18]:
import re
from collections import defaultdict

ETAPA_SIMULACION = "actual"  # o "pre_mundial"
N_SIMULACIONES = 3000
RUTA_CALENDARIO_JSON = DIR_RAIZ / "data" / "raw" / "wc2026_calendario.json"
MAPEO_NOMBRES_JSON = {"Bosnia & Herzegovina": "Bosnia and Herzegovina", "USA": "United States"}


def checkpoint_mas_reciente(dir_checkpoints: Path) -> str:
    """El checkpoint con la fecha de entrenamiento (dato, no reloj de guardado)
    mas avanzada, excluyendo `pre_mundial` -- por definicion, el mas antiguo."""
    candidatos = []
    for carpeta in dir_checkpoints.iterdir():
        if not carpeta.is_dir() or carpeta.name in ("pre_mundial",):
            continue
        metadata_carpeta = json.loads((carpeta / "metadata.json").read_text())
        candidatos.append((metadata_carpeta["ultima_fecha_entrenamiento"], carpeta.name))
    if not candidatos:
        return "pre_mundial"
    return max(candidatos)[1]


nombre_checkpoint_simulacion = "pre_mundial" if ETAPA_SIMULACION == "pre_mundial" else checkpoint_mas_reciente(DIR_MODELOS / "checkpoints")
RUTA_CHECKPOINT_SIMULACION = DIR_MODELOS / "checkpoints" / nombre_checkpoint_simulacion
metadata_simulacion = json.loads((RUTA_CHECKPOINT_SIMULACION / "metadata.json").read_text())
modelo_local_sim = joblib.load(RUTA_CHECKPOINT_SIMULACION / "modelo_goles_local.joblib")
modelo_visitante_sim = joblib.load(RUTA_CHECKPOINT_SIMULACION / "modelo_goles_visitante.joblib")

FECHA_CORTE_SNAPSHOTS = pd.Timestamp(metadata_simulacion["ultima_fecha_entrenamiento"])
USAR_RESULTADOS_REALES = ETAPA_SIMULACION == "actual"
familia_simulacion = metadata_simulacion["familia"]
print(f"Modo: '{ETAPA_SIMULACION}' -- checkpoint '{nombre_checkpoint_simulacion}' "
      f"(familia '{familia_simulacion}'), datos hasta {FECHA_CORTE_SNAPSHOTS.date()}.")

Modo: 'actual' -- checkpoint 'post_dieciseisavos' (familia 'xgboost'), datos hasta 2026-07-03.


### Reconstruir el cuadro como grafo de dependencias

Los cruces de octavos en adelante se normalizan a referencias `W<num>`/`L<num>` buscando en
qué partido anterior quedó cada selección como ganadora o perdedora REAL -- esto es puramente
estructural (identifica de qué partido depende cada cruce, algo fijo por el sorteo del propio
Mundial) y es igual en los dos modos. Lo que SÍ depende del modo es si un partido concreto se
da por "ya decidido" (`es_conocido`): en `pre_mundial` nunca, ni siquiera los que ya se jugaron
de verdad -- ese es precisamente el punto de esa vista.

In [19]:
with open(RUTA_CALENDARIO_JSON, encoding="utf-8") as f:
    calendario_sim = json.load(f)["matches"]
partidos_json_sim = {m["num"]: m for m in calendario_sim if "num" in m}


def ganador_real(partido: dict) -> str | None:
    marcador = partido.get("score")
    if marcador is None:
        return None
    for clave in ("p", "et", "ft"):
        if clave in marcador:
            g1, g2 = marcador[clave]
            if g1 != g2:
                return partido["team1"] if g1 > g2 else partido["team2"]
    return None


equipo_a_num_dieciseisavo = {}
for num, partido in partidos_json_sim.items():
    if num > 88:
        continue
    ganador = ganador_real(partido)
    if ganador:
        equipo_a_num_dieciseisavo[MAPEO_NOMBRES_JSON.get(ganador, ganador)] = num


def normalizar_referencia(referencia: str) -> str:
    equipo = MAPEO_NOMBRES_JSON.get(referencia, referencia)
    return f"W{equipo_a_num_dieciseisavo[equipo]}" if equipo in equipo_a_num_dieciseisavo else equipo


bracket_sim = {}
for num, partido in partidos_json_sim.items():
    if num <= 88:
        referencia_a = MAPEO_NOMBRES_JSON.get(partido["team1"], partido["team1"])
        referencia_b = MAPEO_NOMBRES_JSON.get(partido["team2"], partido["team2"])
    else:
        referencia_a = normalizar_referencia(partido["team1"])
        referencia_b = normalizar_referencia(partido["team2"])
    bracket_sim[num] = {
        "fecha": pd.Timestamp(partido["date"]),
        "referencia_a": referencia_a, "referencia_b": referencia_b,
        "es_conocido": USAR_RESULTADOS_REALES and ganador_real(partido) is not None,
        "ganador_real": MAPEO_NOMBRES_JSON.get(ganador_real(partido), ganador_real(partido)),
    }

bracket_sim_ordenado = sorted(bracket_sim.items())
n_conocidos = sum(p["es_conocido"] for _, p in bracket_sim_ordenado)
print(f"{len(bracket_sim_ordenado)} partidos en el cuadro -- {n_conocidos} ya jugados de verdad "
      f"(fijos, no se simulan) en modo '{ETAPA_SIMULACION}'.")

32 partidos en el cuadro -- 13 ya jugados de verdad (fijos, no se simulan) en modo 'actual'.


### Snapshot y H2H de cada selección, hasta `FECHA_CORTE_SNAPSHOTS`

Misma idea que 4.2-4.5, pero por selección en vez de por partido: elo/forma/racha de la
ÚLTIMA fila jugada de cada equipo antes del corte, y el historial cara a cara agregado de
cada pareja -- todo hasta la fecha que le corresponda al modo elegido.

In [20]:
df_train_sim = df[(df["fecha"] <= FECHA_CORTE_SNAPSHOTS) & df["jugado"]].copy()


def snapshot_equipo(equipo: str, df_train: pd.DataFrame) -> dict:
    es_local = df_train["equipo_local"] == equipo
    es_visitante = df_train["equipo_visitante"] == equipo
    ultima_local = df_train.loc[es_local, "fecha"].max() if es_local.any() else pd.Timestamp.min
    ultima_visitante = df_train.loc[es_visitante, "fecha"].max() if es_visitante.any() else pd.Timestamp.min

    if ultima_local >= ultima_visitante:
        fila = df_train.loc[es_local].sort_values("fecha").iloc[-1]
        sufijo = "_local"
    else:
        fila = df_train.loc[es_visitante].sort_values("fecha").iloc[-1]
        sufijo = "_visitante"

    return {
        "elo": fila[f"elo_actual{sufijo}"], "tendencia_elo": fila[f"tendencia_elo{sufijo}"],
        "forma_gf_5": fila[f"prom_gf_5{sufijo}"], "forma_gf_10": fila[f"prom_gf_10{sufijo}"],
        "racha_5": fila[f"racha_puntos_5{sufijo}"], "racha_10": fila[f"racha_puntos_10{sufijo}"],
        "ultima_fecha": max(ultima_local, ultima_visitante),
    }


def calcular_h2h_snapshot(equipos: list[str], df_train: pd.DataFrame) -> dict[tuple[str, str], tuple[float, float]]:
    resultados: dict[tuple[str, str], tuple[float, float]] = {}
    jugados = df_train[df_train["jugado"]]
    for i, x in enumerate(equipos):
        for y in equipos[i + 1:]:
            previos = jugados[((jugados["equipo_local"] == x) & (jugados["equipo_visitante"] == y))
                               | ((jugados["equipo_local"] == y) & (jugados["equipo_visitante"] == x))]
            if previos.empty:
                resultados[(x, y)] = (1.0, 0.0)
                resultados[(y, x)] = (1.0, 0.0)
                continue
            puntos_x, puntos_y, difs_x, difs_y = [], [], [], []
            for fila in previos.itertuples(index=False):
                gx, gy = (fila.goles_local, fila.goles_visitante) if fila.equipo_local == x else (fila.goles_visitante, fila.goles_local)
                puntos_x.append(3 if gx > gy else (1 if gx == gy else 0))
                puntos_y.append(3 if gy > gx else (1 if gx == gy else 0))
                difs_x.append(gx - gy)
                difs_y.append(gy - gx)
            resultados[(x, y)] = (float(np.mean(puntos_x)), float(np.mean(difs_x)))
            resultados[(y, x)] = (float(np.mean(puntos_y)), float(np.mean(difs_y)))
    return resultados


equipos_dieciseisavos = sorted({p["referencia_a"] for _, p in bracket_sim_ordenado if not re.match(r"^[WL]\d+$", p["referencia_a"])}
                                 | {p["referencia_b"] for _, p in bracket_sim_ordenado if not re.match(r"^[WL]\d+$", p["referencia_b"])})

snapshots_sim = {equipo: snapshot_equipo(equipo, df_train_sim) for equipo in equipos_dieciseisavos}
h2h_sim = calcular_h2h_snapshot(equipos_dieciseisavos, df_train_sim)
print(f"Snapshot y H2H calculados para {len(snapshots_sim)} selecciones, hasta {FECHA_CORTE_SNAPSHOTS.date()}.")

Snapshot y H2H calculados para 32 selecciones, hasta 2026-07-03.


### Sede de cada partido, corrección de Dixon-Coles, y predicción de un cruce

`es_neutral` se deriva de la sede igual que en el resto del pipeline (Notebook 2). `rho` se
recalibra sobre `df_train_sim` -- distinto según el modo, porque el conjunto de entrenamiento
también lo es.

In [21]:
CIUDAD_A_PAIS_MUNDIAL_2026 = {
    "Mexico City": "Mexico", "Zapopan": "Mexico", "Guadalupe": "Mexico",
    "Toronto": "Canada", "Vancouver": "Canada",
    "Inglewood": "United States", "Santa Clara": "United States",
    "East Rutherford": "United States", "Foxborough": "United States",
    "Philadelphia": "United States", "Houston": "United States",
    "Seattle": "United States", "Atlanta": "United States",
    "Miami Gardens": "United States", "Arlington": "United States",
    "Kansas City": "United States",
}


def _ciudad_desde_sede(sede: str) -> str:
    return sede.split("(")[1].rstrip(")") if "(" in sede else sede


PAIS_SEDE_POR_NUM = {num: CIUDAD_A_PAIS_MUNDIAL_2026[_ciudad_desde_sede(partido["ground"])]
                      for num, partido in partidos_json_sim.items()}

X_train_sim = df_train_sim[FEATURES_MODELO]
lam_hist_sim = np.clip(modelo_local_sim.predict(X_train_sim), 0.01, None)
mu_hist_sim = np.clip(modelo_visitante_sim.predict(X_train_sim), 0.01, None)
candidatos_rho_sim = np.linspace(-0.3, 0.3, 121)
log_vero_sim = [log_verosimilitud_dixon_coles(r, lam_hist_sim, mu_hist_sim,
                                                df_train_sim["goles_local"].to_numpy(),
                                                df_train_sim["goles_visitante"].to_numpy())
                for r in candidatos_rho_sim]
RHO_SIMULACION = float(candidatos_rho_sim[np.argmax(log_vero_sim)])
print(f"rho Dixon-Coles ({ETAPA_SIMULACION}): {RHO_SIMULACION:.4f}")


def construir_features_simulacion(equipo_a: str, equipo_b: str, fecha: pd.Timestamp, es_neutral: bool,
                                    ultimo_partido: dict) -> pd.DataFrame:
    snap_a, snap_b = snapshots_sim[equipo_a], snapshots_sim[equipo_b]
    h2h_puntos, h2h_dif_goles = h2h_sim[(equipo_a, equipo_b)]
    return pd.DataFrame([{
        "elo_diff": snap_a["elo"] - snap_b["elo"],
        "tendencia_elo_local": snap_a["tendencia_elo"], "tendencia_elo_visitante": snap_b["tendencia_elo"],
        "dif_forma_gf_5": snap_a["forma_gf_5"] - snap_b["forma_gf_5"],
        "dif_forma_gf_10": snap_a["forma_gf_10"] - snap_b["forma_gf_10"],
        "dif_racha_5": snap_a["racha_5"] - snap_b["racha_5"], "dif_racha_10": snap_a["racha_10"] - snap_b["racha_10"],
        "dias_descanso_local": (fecha - ultimo_partido[equipo_a]).days,
        "dias_descanso_visitante": (fecha - ultimo_partido[equipo_b]).days,
        "es_neutral": es_neutral, "h2h_puntos_prom": h2h_puntos, "h2h_dif_goles_prom": h2h_dif_goles,
    }])[FEATURES_MODELO]


def matriz_probabilidad_conjunta_sim(lam_a: float, lam_b: float, max_goles: int = MAX_GOLES_MATRIZ) -> np.ndarray:
    goles = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(goles, lam_a), poisson.pmf(goles, lam_b))
    matriz[0, 0] *= 1 - lam_a * lam_b * RHO_SIMULACION
    matriz[0, 1] *= 1 + lam_a * RHO_SIMULACION
    matriz[1, 0] *= 1 + lam_b * RHO_SIMULACION
    matriz[1, 1] *= 1 - RHO_SIMULACION
    matriz = np.clip(matriz, 0, None)
    return matriz / matriz.sum()


cache_lambdas_sim: dict = {}


def predecir_lambda_sim(equipo_a: str, equipo_b: str, fecha: pd.Timestamp, es_neutral: bool,
                          ultimo_partido: dict) -> tuple[float, float]:
    clave = (equipo_a, equipo_b, fecha, es_neutral, ultimo_partido[equipo_a], ultimo_partido[equipo_b])
    if clave not in cache_lambdas_sim:
        X = construir_features_simulacion(equipo_a, equipo_b, fecha, es_neutral, ultimo_partido)
        lam_a = float(np.clip(modelo_local_sim.predict(X), 0.01, None)[0])
        lam_b = float(np.clip(modelo_visitante_sim.predict(X), 0.01, None)[0])
        cache_lambdas_sim[clave] = (lam_a, lam_b)
    return cache_lambdas_sim[clave]


PATRON_PLACEHOLDER = re.compile(r"^([WL])(\d+)$")


def resolver_equipo(referencia: str, ganadores: dict, perdedores: dict) -> str:
    m = PATRON_PLACEHOLDER.match(referencia)
    if m is None:
        return referencia
    tipo, num = m.group(1), int(m.group(2))
    return ganadores[num] if tipo == "W" else perdedores[num]

rho Dixon-Coles (actual): -0.0450


### Una simulación completa del cuadro

Para un partido `es_conocido` (ya jugado de verdad, y el modo lo respeta): se usa el ganador
REAL directamente, sin gastar ningún sorteo -- no tiene sentido "simular" un resultado que ya
se conoce. Para el resto: sorteo de Poisson con las lambdas predichas, y un empate a 90
minutos se decide por sorteo ponderado según la matriz conjunta (no por el criterio
determinista de 4.6 -- aquí interesa que la incertidumbre se propague de verdad).

In [22]:
def simular_torneo_una_vez(rng: np.random.Generator) -> tuple[dict, dict]:
    ultimo_partido = {equipo: snap["ultima_fecha"] for equipo, snap in snapshots_sim.items()}
    ganadores, perdedores, marcadores = {}, {}, {}

    for num, partido in bracket_sim_ordenado:
        equipo_a = resolver_equipo(partido["referencia_a"], ganadores, perdedores)
        equipo_b = resolver_equipo(partido["referencia_b"], ganadores, perdedores)
        fecha = partido["fecha"]

        if partido["es_conocido"]:
            ganador = partido["ganador_real"]
            perdedores[num] = equipo_b if ganador == equipo_a else equipo_a
            ganadores[num] = ganador
            marcadores[num] = {"equipo_a": equipo_a, "equipo_b": equipo_b, "ganador": ganador, "jugado": True}
            ultimo_partido[equipo_a] = fecha
            ultimo_partido[equipo_b] = fecha
            continue

        pais_sede = PAIS_SEDE_POR_NUM[num]
        if equipo_b == pais_sede and equipo_a != pais_sede:
            equipo_a, equipo_b = equipo_b, equipo_a
        es_neutral = equipo_a != pais_sede

        lam_a, lam_b = predecir_lambda_sim(equipo_a, equipo_b, fecha, es_neutral, ultimo_partido)
        goles_a, goles_b = int(rng.poisson(lam_a)), int(rng.poisson(lam_b))

        if goles_a != goles_b:
            ganador = equipo_a if goles_a > goles_b else equipo_b
        else:
            matriz = matriz_probabilidad_conjunta_sim(lam_a, lam_b)
            prob_a = np.tril(matriz, -1).sum()
            prob_b = np.triu(matriz, 1).sum()
            ganador = equipo_a if rng.random() < prob_a / (prob_a + prob_b) else equipo_b

        perdedores[num] = equipo_b if ganador == equipo_a else equipo_a
        ganadores[num] = ganador
        marcadores[num] = {"equipo_a": equipo_a, "equipo_b": equipo_b, "lam_a": lam_a, "lam_b": lam_b,
                            "goles_a": goles_a, "goles_b": goles_b, "ganador": ganador, "jugado": False}
        ultimo_partido[equipo_a] = fecha
        ultimo_partido[equipo_b] = fecha

    return ganadores, marcadores


rng_ejemplo = np.random.default_rng(SEMILLA)
ganadores_ejemplo, _ = simular_torneo_una_vez(rng_ejemplo)
print("Campeón de esta simulación de ejemplo:", ganadores_ejemplo[104])

Campeón de esta simulación de ejemplo: England


### Monte Carlo: repetir la simulación miles de veces

In [23]:
ETAPAS = [
    ("alcanza_octavos", range(73, 89)),
    ("alcanza_cuartos", range(89, 97)),
    ("alcanza_semis", range(97, 101)),
    ("alcanza_final", range(101, 103)),
    ("campeon", range(104, 105)),
]

conteo_etapas = defaultdict(lambda: defaultdict(int))
rng = np.random.default_rng(SEMILLA)
for _ in range(N_SIMULACIONES):
    ganadores, _ = simular_torneo_una_vez(rng)
    for nombre_etapa, rango in ETAPAS:
        for num in rango:
            conteo_etapas[ganadores[num]][nombre_etapa] += 1

print(f"{N_SIMULACIONES:,} simulaciones completas del cuadro (modo '{ETAPA_SIMULACION}').")

filas_probabilidades = []
for equipo in equipos_dieciseisavos:
    fila = {"seleccion": equipo}
    for nombre_etapa, _ in ETAPAS:
        fila[nombre_etapa] = conteo_etapas[equipo][nombre_etapa] / N_SIMULACIONES
    filas_probabilidades.append(fila)

df_probabilidades_sim = pd.DataFrame(filas_probabilidades).sort_values("campeon", ascending=False).reset_index(drop=True)
print(f"\n=== Probabilidad de cada selección de alcanzar cada ronda (modo '{ETAPA_SIMULACION}') ===")
print(df_probabilidades_sim.round(4).to_string(index=False))

ruta_probabilidades_sim = DIR_RESULTS / f"simulacion_probabilidades_{ETAPA_SIMULACION}.csv"
df_probabilidades_sim.to_csv(ruta_probabilidades_sim, index=False)
print(f"\nGuardado: {ruta_probabilidades_sim}")

3,000 simulaciones completas del cuadro (modo 'actual').

=== Probabilidad de cada selección de alcanzar cada ronda (modo 'actual') ===
             seleccion  alcanza_octavos  alcanza_cuartos  alcanza_semis  alcanza_final  campeon
             Argentina           0.9470           0.8747         0.7023         0.4173   0.2577
                France           1.0000           0.8630         0.6880         0.4140   0.2287
                 Spain           1.0000           0.6793         0.5220         0.3027   0.2073
               England           1.0000           0.5997         0.3270         0.1720   0.0757
                Brazil           1.0000           0.6093         0.3620         0.1760   0.0667
              Portugal           1.0000           0.3207         0.1957         0.0660   0.0277
               Morocco           1.0000           0.7550         0.2153         0.0940   0.0273
              Colombia           0.9250           0.6033         0.1770         0.0733   0.0250


## Resumen final

- **4.1-4.12**: pipeline de predicción partido a partido, con reentrenamiento walk-forward
  honesto (evalúa antes de conocer el resultado, incorpora después).
- **4.13**: simulación Montecarlo del cuadro completo, con dos vistas posibles
  (`ETAPA_SIMULACION`): `pre_mundial` (el día 0, comparable con lo que pasó de verdad) o
  `actual` (con todo lo jugado hasta hoy, para probabilidades actualizadas por selección).
- Siguiente paso: **Notebook 5**, el cuadro visual con banderas (usa el checkpoint más
  reciente igual que la vista `actual` de aquí, pero con predicción determinista en cascada en
  vez de miles de simulaciones).